In [1]:
import pickle

# Open the pickle file in binary read mode
with open('old_perturb_results/use_g_result/gradient_comparison_norm2modified_exponax_nu0.0005.pkl', 'rb') as file:
    # Load the data from the pickle file
    data = pickle.load(file)

In [2]:
data.keys()

dict_keys([('norm_2', 'index_0', 'numsteps_5', 'epsilon_0.3200', 'alpha_0.20000'), ('norm_2', 'index_0', 'numsteps_5', 'epsilon_3.2000', 'alpha_2.00000'), ('norm_2', 'index_0', 'numsteps_5', 'epsilon_32.0000', 'alpha_20.00000'), ('norm_2', 'index_0', 'numsteps_5', 'epsilon_160.0000', 'alpha_100.00000'), ('norm_2', 'index_0', 'numsteps_5', 'epsilon_320.0000', 'alpha_200.00000'), ('norm_2', 'index_0', 'numsteps_10', 'epsilon_0.3200', 'alpha_0.10000'), ('norm_2', 'index_0', 'numsteps_10', 'epsilon_3.2000', 'alpha_1.00000'), ('norm_2', 'index_0', 'numsteps_10', 'epsilon_32.0000', 'alpha_10.00000'), ('norm_2', 'index_0', 'numsteps_10', 'epsilon_160.0000', 'alpha_50.00000'), ('norm_2', 'index_0', 'numsteps_10', 'epsilon_320.0000', 'alpha_100.00000'), ('norm_2', 'index_0', 'numsteps_20', 'epsilon_0.3200', 'alpha_0.05000'), ('norm_2', 'index_0', 'numsteps_20', 'epsilon_3.2000', 'alpha_0.50000'), ('norm_2', 'index_0', 'numsteps_20', 'epsilon_32.0000', 'alpha_5.00000'), ('norm_2', 'index_0', 'nu

In [26]:

import scipy.stats as stats
from collections import defaultdict
import numpy as np
import pandas as pd

def analyze_gradient_groups(data):
    """
    Groups data by all keys except index, then performs t-tests comparing final losses.
    
    Parameters:
    - data: The nested dictionary containing all experiment results
    
    Returns:
    - Dictionary containing grouped results and t-test outcomes
    """
    # Step 1: Group data by all keys except index
    groups = defaultdict(list)
    
    for key_tuple in data.keys():
        # Extract all parts of the key except index
        norm_type, index_part, numsteps, epsilon, alpha = key_tuple
        group_key = (norm_type, numsteps, epsilon, alpha)
        
        # Store the index and corresponding data
        groups[group_key].append((index_part, data[key_tuple]))
    
    # Step 2: Process each group and perform t-tests
    results = {}
    
    for group_key, group_data in groups.items():
        # Only process groups with at least 2 samples
        if len(group_data) < 2:
            continue
            
        # Extract final step losses for with_solver and without_solver
        with_solver_losses = []
        without_solver_losses = []
        
        for index_part, experiment_data in group_data:
            # Get the final step (step_99)
            num = len(experiment_data['step_update']) - 1
            final_step = experiment_data['step_update'][f'step_{num}']
            
            # Get the 'after' losses
            with_solver_loss = final_step['loss_metrics']['after']['with_solver']['combined_loss']
            without_solver_loss = final_step['loss_metrics']['after']['with_solver_on_without']['combined_loss']
            
            with_solver_losses.append(with_solver_loss)
            without_solver_losses.append(without_solver_loss)
        
        # Perform paired t-test (since these are paired samples - same input index)
        t_stat, p_value = stats.ttest_rel(with_solver_losses, without_solver_losses)
        
        # Calculate mean differences
        mean_diff = np.mean(np.array(with_solver_losses) - np.array(without_solver_losses))
        abs_mean_diff = np.mean(np.abs(np.array(with_solver_losses) - np.array(without_solver_losses)))
        
        # Store results
        results[group_key] = {
            # 'n_samples': len(group_data),
            # 'with_solver_mean': np.mean(with_solver_losses),
            # 'without_solver_mean': np.mean(without_solver_losses),
            # 'mean_difference': mean_diff,
            # 'abs_mean_difference': abs_mean_diff,
            # 't_statistic': t_stat,
            'p_value': p_value,
            # 'significant_at_0.05': p_value < 0.05,
            # 'with_solver_losses': with_solver_losses,
            # 'without_solver_losses': without_solver_losses
        }
    
    return results

def gen_df(data):
    index_values = []
    column_values = []
    p_values = []

    # Process each key in the input data
    for key in data:
        norm = key[0].split('_')[1]
        num_step = float(key[1].split('_')[1])
        epsilon = float(key[2].split('_')[1])
        # alpha = key[3].split('_')[1]
        
        index_values.append((norm, num_step))
        column_values.append((epsilon, ))
        p_values.append(data[key]['p_value'])

    # Create MultiIndex for rows and columns
    row_index = pd.MultiIndex.from_tuples(sorted(list(set(index_values))), names=['norm', 'num_step'])
    col_index = pd.MultiIndex.from_tuples(sorted(list(set(column_values))), names=['epsilon', ])

    # Create empty DataFrame with sorted indices
    df = pd.DataFrame(index=row_index, columns=col_index)

    # Fill the DataFrame
    for (norm, num_step), (epsilon, ), p in zip(index_values, column_values, p_values):
        df.loc[(norm, num_step), (epsilon, )] = p

    # Sort both axes to ensure optimal performance
    df = df.sort_index(axis=0).sort_index(axis=1)

    return df

In [13]:
raw_data = analyze_gradient_groups(data)

In [16]:
norm_2_data = {key:val for key,val in raw_data.items() if "norm_2" in key}
norm_inf_data = {key:val for key,val in raw_data.items() if "norm_inf" in key}

In [27]:
gen_df(norm_2_data)

epsilon          0.32      3.20      32.00     160.00    320.00
norm num_step                                                  
2    5.0       0.000002  0.000003  0.576687  0.672307  0.000334
     10.0      0.000002  0.000003  0.370553  0.025479  0.000052
     20.0      0.000002  0.000003  0.012565  0.009298  0.000135
     50.0      0.000002  0.000003  0.007221  0.058539       0.0
     100.0     0.000002  0.000003  0.007221  0.000068  0.000001

In [28]:
gen_df(norm_inf_data)

epsilon           0.001     0.010     0.100     0.500     1.000
norm num_step                                                  
inf  5.0       0.024665   0.03223  0.024433  0.031011  0.438696
     10.0      0.024809  0.030816  0.028542  0.018932  0.800475
     20.0      0.025145  0.029923  0.040934  0.032119  0.529914
     50.0      0.025417  0.029044  0.027876  0.019122  0.759447
     100.0     0.025464  0.028881  0.027025  0.021433  0.713721